In [ ]:
# ============================================================
# Project  : Olist E-Commerce Behavioral Analysis
# Author   : Supriya
# Tool     : Python — Pandas only
# Dataset  : Brazilian E-Commerce (Olist)
#            Source: Kaggle
# Tables   : 8 datasets | 100K+ orders | 2016–2018
# ============================================================

In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
orders      = pd.read_csv(r"C:\olist_ecommerce\olist_orders_dataset.csv")
payments    = pd.read_csv(r"C:\olist_ecommerce\olist_order_payments_dataset.csv")
reviews     = pd.read_csv(r"C:\olist_ecommerce\olist_order_reviews_dataset.csv")
order_items = pd.read_csv(r"C:\olist_ecommerce\olist_order_items_dataset.csv")
products    = pd.read_csv(r"C:\olist_ecommerce\olist_products_dataset.csv")
sellers     = pd.read_csv(r"C:\olist_ecommerce\olist_sellers_dataset.csv")
customers   = pd.read_csv(r"C:\olist_ecommerce\olist_customers_dataset.csv")
cat_trans   = pd.read_csv(r"C:\olist_ecommerce\product_category_name_translation.csv")

print("orders     :", orders.shape)
print("payments   :", payments.shape)
print("reviews    :", reviews.shape)
print("order_items:", order_items.shape)
print("products   :", products.shape)
print("sellers    :", sellers.shape)
print("customers  :", customers.shape)
print("cat_trans  :", cat_trans.shape)

orders     : (99441, 8)
payments   : (103886, 5)
reviews    : (99224, 7)
order_items: (112650, 7)
products   : (32951, 9)
sellers    : (3095, 4)
customers  : (99441, 5)
cat_trans  : (71, 2)


In [3]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [4]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [5]:
orders.duplicated().sum()

np.int64(0)

In [6]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [7]:
orders_delivered = orders[orders["order_status"] == "delivered"].copy()

orders_delivered["year"]          = orders_delivered["order_purchase_timestamp"].dt.year
orders_delivered["month"]         = orders_delivered["order_purchase_timestamp"].dt.month
orders_delivered["year_month"]    = orders_delivered["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders_delivered["delivery_days"] = (
    orders_delivered["order_delivered_customer_date"] -
    orders_delivered["order_approved_at"]
).dt.days

orders_delivered[["year", "month", "year_month", "delivery_days"]].head()

,year,month,year_month,delivery_days
0,2017,10,2017-10,8.0
1,2018,7,2018-07,12.0
2,2018,8,2018-08,9.0
3,2017,11,2017-11,13.0
4,2018,2,2018-02,2.0


In [8]:
pd.DataFrame({
    "metric": ["Total Orders", "Delivered Orders", "Unique Customers",
                "Unique Products", "Unique Sellers", "Date From", "Date To"],
    "value": [
        orders["order_id"].nunique(),
        orders_delivered["order_id"].nunique(),
        customers["customer_unique_id"].nunique(),
        products["product_id"].nunique(),
        sellers["seller_id"].nunique(),
        orders["order_purchase_timestamp"].min().date(),
        orders["order_purchase_timestamp"].max().date(),
    ]
})

,metric,value
0,Total Orders,99441
1,Delivered Orders,96478
2,Unique Customers,96096
3,Unique Products,32951
4,Unique Sellers,3095
5,Date From,2016-09-04
6,Date To,2018-10-17


In [9]:
# ══════════════════════════════════════════════════════════════
# 4. PAYMENT ANALYSIS
# ══════════════════════════════════════════════════════════════

payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [10]:
payments.groupby("payment_type").agg(
    count       = ("order_id",      "count"),
    total_value = ("payment_value", "sum"),
    avg_value   = ("payment_value", "mean"),
).sort_values("total_value", ascending=False).round(2)

,count,total_value,avg_value
payment_type,,,
credit_card,76795,12542084.19,163.32
boleto,19784,2869361.27,145.03
voucher,5775,379436.87,65.70
debit_card,1529,217989.79,142.57
not_defined,3,0.00,0.00


In [11]:
payments[payments["payment_installments"] > 1]["payment_type"].value_counts()

payment_type
credit_card    51338
Name: count, dtype: int64

In [12]:
# ══════════════════════════════════════════════════════════════
# 5. REVENUE BY PRODUCT CATEGORY
# ══════════════════════════════════════════════════════════════

items_products = order_items.merge(products, on="product_id")
items_products = items_products.merge(cat_trans, on="product_category_name", how="left")
items_pay      = items_products.merge(payments, on="order_id")
items_pay      = items_pay.merge(orders_delivered[["order_id"]], on="order_id")

items_pay.shape

(115035, 20)

In [13]:
(
    items_pay.groupby("product_category_name_english")
    .agg(
        num_orders = ("order_id",      "nunique"),
        revenue    = ("payment_value", "sum"),
        avg_order  = ("payment_value", "mean"),
    )
    .sort_values("revenue", ascending=False)
    .head(15)
    .round(2)
)

,num_orders,revenue,avg_order
product_category_name_english,,,
bed_bath_table,9272,1692714.28,145.30
health_beauty,8646,1620684.04,166.07
computers_accessories,6530,1549372.59,196.17
furniture_decor,6307,1394466.93,162.96
watches_gifts,5495,1387362.45,228.75
sports_leisure,7530,1349446.93,154.52
housewares,5743,1069787.97,149.16
auto,3810,833745.67,194.62
garden_tools,3448,810614.93,181.59


In [14]:
# ══════════════════════════════════════════════════════════════
# 6. MONTHLY ORDER TREND
# ══════════════════════════════════════════════════════════════

orders_delivered.groupby("year_month").agg(
    num_orders       = ("order_id",    "count"),
    unique_customers = ("customer_id", "nunique"),
).reset_index()

,year_month,num_orders,unique_customers
0,2016-09,1,1
1,2016-10,265,265
2,2016-12,1,1
3,2017-01,750,750
4,2017-02,1653,1653
5,2017-03,2546,2546
6,2017-04,2303,2303
7,2017-05,3546,3546
8,2017-06,3135,3135
9,2017-07,3872,3872


In [15]:
orders_delivered.groupby("year").agg(
    num_orders       = ("order_id",    "count"),
    unique_customers = ("customer_id", "nunique"),
).round(2)

,num_orders,unique_customers
year,,
2016,267,267
2017,43428,43428
2018,52783,52783


In [16]:
# ══════════════════════════════════════════════════════════════
# 7. DELIVERY PERFORMANCE
# ══════════════════════════════════════════════════════════════

orders_delivered["delivery_days"].describe().round(2)

count    96456.00
mean        11.64
std          9.52
min         -7.00
25%          6.00
50%          9.00
75%         15.00
max        208.00
Name: delivery_days, dtype: float64

In [17]:
cust_orders = orders_delivered.merge(customers, on="customer_id")

cust_orders.groupby("customer_state")["delivery_days"].mean() \
    .sort_values() \
    .round(1) \
    .reset_index() \
    .rename(columns={"delivery_days": "avg_delivery_days"}) \
    .head(10)

,customer_state,avg_delivery_days
0,SP,7.9
1,PR,11.0
2,MG,11.1
3,DF,12.1
4,SC,14.0
5,RS,14.3
6,RJ,14.4
7,GO,14.6
8,MS,14.7
9,ES,14.9


In [18]:
# ══════════════════════════════════════════════════════════════
# 8. REVIEW SCORE ANALYSIS
# ══════════════════════════════════════════════════════════════

reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [19]:
reviews["review_score"].value_counts(normalize=True).sort_index().round(3)

review_score
1    0.115
2    0.032
3    0.082
4    0.193
5    0.578
Name: proportion, dtype: float64

In [20]:
reviews_orders = reviews.merge(orders_delivered[["order_id", "year_month"]], on="order_id")

reviews_orders.groupby("year_month")["review_score"].mean().round(2).reset_index()

,year_month,review_score
0,2016-09,1.00
1,2016-10,4.02
2,2016-12,5.00
3,2017-01,4.21
4,2017-02,4.20
5,2017-03,4.18
6,2017-04,4.14
7,2017-05,4.24
8,2017-06,4.22
9,2017-07,4.25


In [21]:
# ══════════════════════════════════════════════════════════════
# 9. SELLER ANALYSIS
# ══════════════════════════════════════════════════════════════

(
    items_pay.merge(sellers, on="seller_id")
    .groupby("seller_id")
    .agg(
        num_orders = ("order_id",      "nunique"),
        revenue    = ("payment_value", "sum"),
        avg_order  = ("payment_value", "mean"),
    )
    .sort_values("revenue", ascending=False)
    .head(10)
    .round(2)
)

,num_orders,revenue,avg_order
seller_id,,,
7c67e1448b00f6e969d365cea6b010ab,973,505437.16,350.27
1025f0e2d44d7041d6cf58b6550e0bfa,910,306000.35,210.45
4a3ca9315b744ce9f8e9374361493884,1772,295830.76,141.28
1f50f920176fa81dab994f9023523100,1399,289861.38,144.79
53243585a1d6dc2643021fd1853d8905,348,279843.42,655.37
da8622b14eb17ae2831f4ac5b9dab84a,1311,271733.78,166.40
4869f7a5dfa277a7dca6462dcf3b52b2,1124,261532.48,222.01
955fee9216a65b617aa5c0531780ce60,1261,232136.06,154.65
fa1c13f2614d7b5c4749cbc52fecda94,578,203262.00,337.64


In [22]:
sellers["seller_state"].value_counts().head(10)

seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64

In [23]:
# ══════════════════════════════════════════════════════════════
# 10. CUSTOMER & STATE ANALYSIS
# ══════════════════════════════════════════════════════════════

cust_pay = cust_orders.merge(payments, on="order_id")

cust_pay.groupby("customer_state").agg(
    num_customers = ("customer_unique_id", "nunique"),
    num_orders    = ("order_id",           "nunique"),
    total_revenue = ("payment_value",      "sum"),
    avg_spend     = ("payment_value",      "mean"),
).sort_values("total_revenue", ascending=False).round(2).head(10)

,num_customers,num_orders,total_revenue,avg_spend
customer_state,,,,
SP,39155,40500,5770266.19,136.39
RJ,11917,12350,2055690.45,158.08
MG,11001,11354,1819277.61,154.12
RS,5168,5345,861802.40,155.45
PR,4769,4923,781919.55,152.45
SC,3449,3546,595208.40,162.58
BA,3158,3256,591270.60,169.76
DF,2019,2080,346146.17,161.60
GO,1895,1957,334294.22,163.31


In [24]:
# ══════════════════════════════════════════════════════════════
# 11. RFM ANALYSIS
# ══════════════════════════════════════════════════════════════

rfm_base = (
    orders_delivered
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    .merge(payments, on="order_id")
)

snapshot_date = rfm_base["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

rfm = (
    rfm_base.groupby("customer_unique_id")
    .agg(
        last_purchase = ("order_purchase_timestamp", "max"),
        frequency     = ("order_id",                "nunique"),
        monetary      = ("payment_value",           "sum"),
    )
    .reset_index()
)

rfm["recency"] = (snapshot_date - rfm["last_purchase"]).dt.days
rfm = rfm.drop(columns=["last_purchase"])
rfm = rfm[rfm["monetary"].notnull()]

rfm.describe().round(2)

,frequency,monetary,recency
count,93357.00,93357.00,93357.00
mean,1.03,165.20,237.94
std,0.21,226.31,152.58
min,1.00,9.59,1.00
25%,1.00,63.06,114.00
50%,1.00,107.78,219.00
75%,1.00,182.56,346.00
max,15.00,13664.08,695.00


In [25]:
rfm["R_score"] = pd.qcut(rfm["recency"],  5, labels=[5, 4, 3, 2, 1])
rfm["F_score"] = rfm["frequency"].apply(lambda x: "2" if x > 1 else "1")
rfm["M_score"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5])

rfm["RFM_segment"] = rfm["R_score"].astype(str) + rfm["F_score"] + rfm["M_score"].astype(str)

rfm.head(10)

,customer_unique_id,frequency,monetary,recency,R_score,F_score,M_score,RFM_segment
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,112,4,1,4,414
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,115,4,1,1,411
2,0000f46a3911fa3c0805444483337064,1,86.22,537,1,1,2,112
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,321,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,288,2,1,4,214
5,0004bd2a26a76fe21f786e4fbd80607f,1,166.98,146,4,1,4,414
6,00050ab1314c0e55a6ca13cf7181fecf,1,35.38,132,4,1,1,411
7,00053a61a98854899e70ed204dd4bafe,1,419.18,183,3,1,5,315
8,0005e1862207bf6ccc02e4228effd9a0,1,150.12,543,1,1,4,114
9,0005ef4cd20d2893f0d9fbd94d3c0d97,1,129.76,170,4,1,3,413


In [26]:
rfm.groupby("F_score").agg(
    customers     = ("customer_unique_id", "count"),
    avg_recency   = ("recency",            "mean"),
    avg_frequency = ("frequency",          "mean"),
    avg_monetary  = ("monetary",           "mean"),
).round(2)

,customers,avg_recency,avg_frequency,avg_monetary
F_score,,,,
1,90556,238.48,1.00,160.76
2,2801,220.29,2.11,308.59


In [27]:
rfm.sort_values("monetary", ascending=False).head(10)

,customer_unique_id,frequency,monetary,recency,R_score,F_score,M_score,RFM_segment
3724,0a0a92112bd4c708ca5fde585afaa872,1,13664.08,334,2,1,5,215
79635,da122df9eeddfedc1dc1f5349a1a690c,2,7571.63,515,1,2,5,125
43168,763c8b1c9c68a0229c42c9fc6f662b93,1,7274.88,46,5,1,5,515
80462,dc4802a71eae9be1dd28f5d788ceb526,1,6929.31,563,1,1,5,115
25436,459bef486812aa25204be022145caa62,1,6922.21,35,5,1,5,515
93080,ff4159b92c40ebe40454e3e6a7c35ed6,1,6726.66,462,1,1,5,115
23411,4007669dec559734d6f53e029e360987,1,6081.54,279,2,1,5,215
87147,eebb5dda148d3893cdaf5b5ca3040ccb,1,4764.34,498,1,1,5,115
26640,48e1ac109decbb87765a3eade6854098,1,4681.78,69,5,1,5,515
73126,c8460e4251689ba205045f3ea17884a1,4,4655.91,22,5,2,5,525


In [28]:
# ══════════════════════════════════════════════════════════════
# 12. REPURCHASE RATE
# ══════════════════════════════════════════════════════════════

repurchase = (
    orders_delivered
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    .groupby("customer_unique_id")
    .agg(order_count=("order_id", "nunique"))
    .reset_index()
)

repurchase["repurchased"] = (repurchase["order_count"] > 1).astype(int)

repurchase["repurchased"].value_counts(normalize=True).round(3)

repurchased
0    0.97
1    0.03
Name: proportion, dtype: float64

In [31]:
cat_repurchase = (
    items_pay
    .merge(orders_delivered[["order_id", "customer_id"]], on="order_id", how="left")
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left")
    .groupby(["customer_unique_id", "product_category_name_english"])
    .agg(orders=("order_id", "nunique"))
    .reset_index()
)

cat_repurchase["repurchased"] = (cat_repurchase["orders"] > 1).astype(int)

(
    cat_repurchase.groupby("product_category_name_english")["repurchased"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .round(3)
    .reset_index()
    .rename(columns={"repurchased": "repurchase_rate"})
)

,product_category_name_english,repurchase_rate
0,arts_and_craftmanship,0.095
1,home_appliances,0.073
2,home_comfort_2,0.043
3,diapers_and_hygiene,0.042
4,furniture_bedroom,0.035
5,fashion_bags_accessories,0.030
6,bed_bath_table,0.027
7,sports_leisure,0.023
8,books_imported,0.020
9,furniture_decor,0.020


In [32]:
# ══════════════════════════════════════════════════════════════
# 13. COHORT RETENTION ANALYSIS
# ══════════════════════════════════════════════════════════════

cohort_base = (
    orders_delivered
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
)

cohort_base["cohort_month"] = (
    cohort_base.groupby("customer_unique_id")["order_purchase_timestamp"]
    .transform("min")
    .dt.to_period("M")
)

cohort_base["order_month"] = cohort_base["order_purchase_timestamp"].dt.to_period("M")
cohort_base["month_gap"]   = (
    cohort_base["order_month"] - cohort_base["cohort_month"]
).apply(lambda x: x.n)

cohort = (
    cohort_base.groupby(["cohort_month", "month_gap"])["customer_unique_id"]
    .nunique()
    .reset_index()
    .rename(columns={"customer_unique_id": "customers"})
)

cohort_size = cohort[cohort["month_gap"] == 0].set_index("cohort_month")["customers"]

cohort["retention_rate"] = (
    cohort.apply(lambda row: row["customers"] / cohort_size[row["cohort_month"]], axis=1)
    .round(3)
)

cohort.pivot_table(
    index="cohort_month",
    columns="month_gap",
    values="retention_rate",
    fill_value=0
).head(12)

month_gap,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20
cohort_month,,,,,,,,,,,,,,,,,,,,
2016-09,1.0,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2016-10,1.0,0.000,0.000,0.000,0.000,0.000,0.004,0.000,0.000,0.004,0.000,0.004,0.000,0.004,0.000,0.004,0.000,0.004,0.008,0.008
2016-12,1.0,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2017-01,1.0,0.003,0.003,0.001,0.004,0.001,0.004,0.001,0.001,0.000,0.004,0.001,0.007,0.004,0.001,0.001,0.003,0.004,0.001,0.000
2017-02,1.0,0.002,0.003,0.001,0.004,0.001,0.002,0.002,0.001,0.002,0.001,0.003,0.001,0.002,0.001,0.001,0.001,0.002,0.000,0.000
2017-03,1.0,0.004,0.004,0.004,0.004,0.002,0.002,0.003,0.003,0.001,0.004,0.001,0.002,0.001,0.002,0.002,0.001,0.001,0.000,0.000
2017-04,1.0,0.006,0.002,0.002,0.003,0.003,0.004,0.003,0.003,0.002,0.003,0.001,0.000,0.000,0.001,0.001,0.001,0.000,0.000,0.000
2017-05,1.0,0.005,0.005,0.003,0.003,0.003,0.004,0.001,0.003,0.003,0.003,0.003,0.002,0.000,0.002,0.002,0.000,0.000,0.000,0.000
2017-06,1.0,0.005,0.004,0.004,0.003,0.004,0.004,0.002,0.001,0.002,0.003,0.004,0.002,0.002,0.002,0.000,0.000,0.000,0.000,0.000
